# 11 — Validação Final

Abre os blocos de teste **uma única vez**, depois de `G7`, `G8` e `G9` aprovados. Reconstrói o pipeline do YAML (treino em `train_ref`), calibra na metade `val_cal` do `val_ref` (mesmo split do notebook 10) e avalia o modelo congelado em três janelas: **Teste A** (`test_ref`, resultado principal), **2020–2022** (`test_pandemic`, isolado) e **Teste B** (`test_ref` + `test_pandemic`). Nada é reajustado.

No final, marca `G9_holdout` como `executado` e registra `test_opened` no manifesto do artefato.

**Ensaio sem abrir o teste:** com `SRAG_ENSAIO=1` o notebook roda de ponta a ponta usando duas metades da `val_meas` no lugar dos blocos de teste. Não lê o teste, não marca o G9, não altera o artefato e grava com o prefixo `11_ensaio_` (que não aciona a guarda de reabertura).

**Pré-requisito:** `pip install -e.` com scikit-learn ≥ 1.6.

In [1]:
from pathlib import Path
import hashlib
import json
import os
import re
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split


def encontrar_raiz_projeto(start: Path | None = None) -> Path:
    atual = (start or Path.cwd()).resolve()
    for candidato in (atual, *atual.parents):
        if (candidato / "src").is_dir() and (candidato / "pyproject.toml").is_file():
            return candidato
    raise RuntimeError(f"Raiz do projeto não encontrada a partir de: {atual}")


ROOT_DIR = encontrar_raiz_projeto()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

try:
    import sklearn.frozen  # noqa: F401
except ImportError as exc:
    raise RuntimeError(
        "Este notebook requer scikit-learn >= 1.6 (FrozenEstimator)."
    ) from exc

from src import config as C
from src import config as env
from src.registro import Run
from src.dados import MANIFEST_NAME, GatePendenteError, load_supervised_artifact, split_partitions
from src.avaliacao import metricas_completas, ic_bootstrap, bootstrap_grupo, auc, average_precision, brier as brier_metric
from src.avaliacao import matriz_confusao, metricas_por_grupo
from src.avaliacao import curva_confiabilidade, ece, mce, brier_decomposicao
from src.features import faixa_etaria as _faixa_etaria
from src.modelos import build_finalista, finalistas_aceitos, resolver_finalista
from src.modelos import calibrar

HASH_LEN                = 16
CALIB_SPLIT_FRAC_PADRAO = 0.5
N_BINS_CALIB            = 10
MIN_N_CALIB_ANO         = 20
N_FN_AMOSTRA            = 20
TAMANHO_LOTE_PREDICAO   = 250_000
LIMITE_LINHAS_COM_FEATURES = 500_000

# Ensaio: roda tudo sem abrir o teste (duas metades da val_meas fazem o papel dos blocos de teste).
ENSAIO       = os.getenv("SRAG_ENSAIO", "0") == "1"
PREFIXO_BASE = "11_ensaio_validacao_final" if ENSAIO else "11_validacao_final"

env.carregar_dotenv()
env.garantir_diretorios()

TABLES_DIR      = C.TABLES_DIR
PREDICTIONS_DIR = C.PREDICTIONS_DIR
TABLES_DIR.mkdir(parents=True, exist_ok=True)
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

print(C.resumo())
if ENSAIO:
    print("=" * 70)
    print("MODO ENSAIO: o teste NÃO será lido, o G9 NÃO será marcado e as saídas levam o prefixo 11_ensaio_.")
    print("=" * 70)

ROOT_DIR          = C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador
DATA_DIR          = C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\data
RAW_DIR           = C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\data\raw
PROCESSED_DIR     = C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\data\processed
ANALYTICAL_DIR    = C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\data\analytical
TREINO_DIR        = C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\data\treino
OUTPUTS_DIR       = C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\outputs
RUNS_DIR          = C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\outputs\runs
LOGS_DIR          = C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\logs
CONFIGS_DIR       = C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\configs
SUPERVISED_YAML   = C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integra

## Verificar gates

`G7`, `G8` e `G9` precisam estar `aprovado` antes de qualquer leitura do teste. O `load_supervised_artifact` reforça `G9_holdout` verificação extra.

In [2]:
GATES_EXIGIDOS = ["G7_calibracao", "G8_limiar", "G9_holdout"]
status_gates   = {g: (C.cfg("gates", g) or {}).get("status") for g in GATES_EXIGIDOS}

display(pd.DataFrame({"gate": list(status_gates), "status": list(status_gates.values())}))

n_acessos_antes = int((C.cfg("gates", "G9_holdout") or {}).get("n_acessos") or 0)

if ENSAIO:
    print("ENSAIO: gates não exigidos (o teste não é aberto).")
else:
    if faltando := [g for g, s in status_gates.items() if s != "aprovado"]:
        raise RuntimeError(
            f"Gates não aprovados: {faltando}. Edite configs/supervised.yaml."
        )
    print(f"Gates aprovados. n_acessos_antes (G9) = {n_acessos_antes}.")

,gate,status
0,G7_calibracao,aprovado
1,G8_limiar,aprovado
2,G9_holdout,pendente


ENSAIO: gates não exigidos (o teste não é aberto).


## Carregar o desenvolvimento e a configuração congelada

Aqui só `train_ref` e `val_ref` são lidos. Os blocos de teste só são materializados na célula de abertura, e só se `G9_holdout` estiver aprovado verificação extra do `load_supervised_artifact`). Finalista, calibração, limiar e variáveis precisam estar congelados no YAML.

In [3]:
COHORT_KIND = os.getenv("SRAG_COHORT_KIND", "final").lower()

# Aqui só o desenvolvimento (train_ref e val_ref). Os blocos de teste são lidos na célula de abertura.
df, manifest = load_supervised_artifact(
    ROOT_DIR,
    expected_kind=COHORT_KIND,
    partitions=("train_ref", "val_ref"),
)

PARTICAO = manifest["partition_column"]
ALVO     = manifest["target_column"]
FEATURES = list(manifest["feature_columns"])

FEATURES_YAML = list(C.cfg("features", "final") or [])
if not FEATURES_YAML:
    raise RuntimeError("features.final vazio — congele as features antes do notebook 11.")
if set(FEATURES) != set(FEATURES_YAML):
    raise RuntimeError(
        "features.final do YAML difere das variáveis do artefato "
        f"(só no YAML: {sorted(set(FEATURES_YAML) - set(FEATURES))}; "
        f"só no artefato: {sorted(set(FEATURES) - set(FEATURES_YAML))}). "
        "Regere o artefato (notebook 02) e rode o notebook 10 de novo."
    )

selecao_cfg = C.cfg("selecao_modelos") or {}
finalista_escolhido = selecao_cfg.get("finalista_escolhido")
if finalista_escolhido is None:
    candidatos = list(selecao_cfg.get("finalistas") or [])
    if ENSAIO and candidatos:
        finalista_escolhido = "logistic" if "logistic" in candidatos else candidatos[0]
        print(f"ENSAIO: finalista_escolhido vazio no YAML; usando {finalista_escolhido!r}.")
    elif len(candidatos) == 1:
        finalista_escolhido = candidatos[0]
    else:
        raise RuntimeError(
            "selecao_modelos.finalista_escolhido não definido e "
            f"finalistas={candidatos}. Congele o finalista no YAML."
        )

calib_cfg    = C.cfg("calibracao") or {}
calib_ativa  = bool(calib_cfg.get("ativa", True))
calib_metodo = calib_cfg.get("metodo")
calib_fonte  = str(calib_cfg.get("fonte") or "validacao")

if calib_ativa and calib_metodo is None:
    if ENSAIO:
        calib_metodo = "sigmoid"
        print("ENSAIO: calibracao.metodo vazio no YAML; usando 'sigmoid'.")
    else:
        raise RuntimeError("calibracao.ativa=true mas calibracao.metodo vazio no YAML.")
if calib_fonte != "validacao":
    raise RuntimeError(f"calibracao.fonte deve ser 'validacao', não {calib_fonte!r}.")

limiar_cfg     = C.cfg("limiar") or {}
limiar_final   = limiar_cfg.get("otimo")
limiar_metrica = limiar_cfg.get("metrica")
if limiar_final is None:
    if ENSAIO:
        limiar_final = 0.5
        print("ENSAIO: limiar.otimo vazio no YAML; usando 0,5.")
    else:
        raise RuntimeError("limiar.otimo vazio no YAML.")
limiar_final_val = float(limiar_final)

proibidas = set(C.cfg("features", "proibidas") or [])
if conflito := set(FEATURES) & proibidas:
    raise RuntimeError(f"features.final contém colunas proibidas: {sorted(conflito)}")

janelas_cfg = C.cfg("avaliacao", "janelas") or {}
JANELAS = {nome: list((cfg or {}).get("blocos") or []) for nome, cfg in janelas_cfg.items()}
if not {"teste_a", "teste_b", "pandemico_isolado"} <= set(JANELAS):
    raise RuntimeError(f"avaliacao.janelas incompleto no YAML: {sorted(JANELAS)}.")
JANELA_PRINCIPAL = next(
    (n for n, cfg in janelas_cfg.items() if (cfg or {}).get("resultado_principal")), "teste_a"
)
BLOCOS_TESTE = list(dict.fromkeys(b for blocos in JANELAS.values() for b in blocos))

display(pd.DataFrame([
    {"chave": "finalista",         "valor": finalista_escolhido},
    {"chave": "calibracao.metodo", "valor": calib_metodo if calib_ativa else "desativada"},
    {"chave": "limiar.otimo",      "valor": limiar_final_val},
    {"chave": "limiar.metrica",    "valor": limiar_metrica},
    {"chave": "features.final",    "valor": len(FEATURES)},
    {"chave": "janelas",           "valor": {n: "+".join(b) for n, b in JANELAS.items()}},
    {"chave": "n_acessos_G9",      "valor": n_acessos_antes},
    {"chave": "ensaio",            "valor": ENSAIO},
]))

,chave,valor
0,finalista,boosting
1,calibracao.metodo,sigmoid
2,limiar.otimo,0.1178
3,limiar.metrica,precision
4,features.final,30
5,janelas,"{'teste_a': 'test_ref', 'teste_b': 'test_ref+t..."
6,n_acessos_G9,0
7,ensaio,True


## Guarda contra reabertura

Se já existe manifesto notebook 11 para o mesmo `dataset_sha256`, o teste foi aberto antes. Reabrir exige `SRAG_ALLOW_TEST_REOPEN=1` e justificativa em `docs/decisoes.md`.

In [4]:
ALLOW_REOPEN = os.getenv("SRAG_ALLOW_TEST_REOPEN", "0") == "1"

reaberturas = []
if ENSAIO:
    print("ENSAIO: a guarda de reabertura não se aplica (o teste não é aberto).")
else:
    for caminho in TABLES_DIR.glob("11_validacao_final_*_manifesto.json"):
        try:
            dados = json.loads(caminho.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            continue
        if dados.get("dataset_sha256") == manifest["dataset_sha256"]:
            reaberturas.append({"arquivo": caminho.name, "em_utc": dados.get("teste_aberto_em_utc")})
    if manifest.get("test_opened"):
        reaberturas.append({"arquivo": "manifesto_supervisionado.json", "em_utc": manifest.get("test_opened_utc")})

    if reaberturas and not ALLOW_REOPEN:
        display(pd.DataFrame(reaberturas))
        raise RuntimeError(
            "Teste já aberto para este dataset_sha256. Defina "
            "SRAG_ALLOW_TEST_REOPEN=1 para reabrir e registre o motivo em docs/decisoes.md."
        )
    print("Guarda OK." if not reaberturas else f"Reabertura autorizada ({len(reaberturas)} registro(s) anterior(es)).")

ENSAIO: a guarda de reabertura não se aplica (o teste não é aberto).


## Reproduzir o split do notebook 10 e calibrar

O limiar congelado foi escolhido na metade `val_meas` do `val_ref`, com o calibrador ajustado na metade `val_cal`. Localizamos o manifesto do notebook 10 compatível e conferimos, **antes de abrir o teste**: o hash do split, as variáveis, o limiar do YAML e o hash da configuração dos modelos (se o YAML dos modelos mudou depois do notebook 10, o notebook 10 precisa rodar de novo). O nome do finalista é resolvido em `src.modelos` (aliases como `hist_gradient_boosting → boosting`); o notebook não mantém cópia do mapeamento.

In [5]:
def _reproduzir_split_val_cal(y_val, frac_cal, seed):
    idx_cal, idx_meas = train_test_split(
        np.arange(len(y_val)),
        test_size=1.0 - frac_cal,
        random_state=seed,
        stratify=y_val.to_numpy(),
    )
    return np.asarray(idx_cal, dtype=np.int64), np.asarray(idx_meas, dtype=np.int64)


def _chave_c5(finalista):
    """Nome do finalista como o notebook 10 o grava em `escolhas` (o que está em selecao_modelos.finalistas)."""
    for nome in (selecao_cfg.get("finalistas") or []):
        if resolver_finalista(nome) == resolver_finalista(finalista):
            return nome
    return finalista


def _localizar_manifesto_c5(dataset_sha256, finalista, metodo, calib_ativa):
    for caminho in sorted(
        TABLES_DIR.glob("10_calibracao_limiar_*_manifesto.json"),
        key=lambda p: p.stat().st_mtime, reverse=True,
    ):
        try:
            dados = json.loads(caminho.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            continue
        if dados.get("dataset_sha256") != dataset_sha256:
            continue
        if calib_ativa:
            escolha = (dados.get("escolhas") or {}).get(_chave_c5(finalista)) or {}
            if escolha.get("metodo") != metodo:
                continue
        return dados, caminho
    return None, None


def _hash_config_modelo(features, finalista):
    # Mesma receita do notebook 10: se o YAML dos modelos mudou depois do notebook 10, o hash muda.
    carga = {"models": C.cfg("models"), "features": sorted(features), "finalista": resolver_finalista(finalista)}
    return hashlib.sha256(json.dumps(carga, sort_keys=True, default=str).encode("utf-8")).hexdigest()[:HASH_LEN]


treino, validacao = split_partitions(df, manifest)

X_train = treino[FEATURES]
y_train = pd.to_numeric(treino[ALVO], errors="coerce").astype("int8")
X_val   = validacao[FEATURES]
y_val   = pd.to_numeric(validacao[ALVO], errors="coerce").astype("int8")

manifesto_c5, manifesto_c5_path = _localizar_manifesto_c5(
    manifest["dataset_sha256"], finalista_escolhido,
    calib_metodo if calib_ativa else None, calib_ativa,
)

if manifesto_c5 is None:
    if ENSAIO:
        print("ENSAIO: manifesto do notebook 10 não encontrado; usando o split padrão (honesto, 50%, semente do YAML).")
    elif calib_ativa:
        raise RuntimeError(
            "Nenhum manifesto notebook 10 compatível (dataset_sha256 + metodo). "
            "Rode o notebook 10 com a configuração atual do YAML antes do notebook 11."
        )

if manifesto_c5 is not None and not ENSAIO:
    # Travas antes de abrir o teste: o que o notebook 10 mediu tem de ser exatamente o que o YAML congelou.
    if set(manifesto_c5.get("feature_columns") or []) != set(FEATURES):
        raise RuntimeError("O notebook 10 foi rodado com outras variáveis: rode o notebook 10 de novo antes do notebook 11.")
    limiar_c5 = ((manifesto_c5.get("escolhas") or {}).get(_chave_c5(finalista_escolhido)) or {}).get("limiar")
    if limiar_c5 is None or abs(float(limiar_c5) - limiar_final_val) > 1e-6:
        raise RuntimeError(
            f"O limiar do manifesto do notebook 10 ({limiar_c5}) não é o do YAML ({limiar_final_val}): "
            "copie o limiar impresso pelo notebook 10 para limiar.otimo ou rode o notebook 10 de novo."
        )
    hash_modelo_c5 = manifesto_c5.get("config_modelo_hash")
    hash_modelo_agora = _hash_config_modelo(FEATURES, finalista_escolhido)
    if hash_modelo_c5 != hash_modelo_agora:
        raise RuntimeError(
            "Os modelos do YAML mudaram depois do notebook 10 (ou o notebook 10 é de uma versão sem registro do hash): "
            f"notebook 10={hash_modelo_c5}, agora={hash_modelo_agora}. Rode o notebook 10 de novo antes do notebook 11."
        )

HONEST_CALIB_SPLIT = bool(manifesto_c5.get("honest_calib_split", False)) if manifesto_c5 else True
CALIB_SPLIT_FRAC   = float(manifesto_c5.get("calib_split_frac", CALIB_SPLIT_FRAC_PADRAO)) if manifesto_c5 else CALIB_SPLIT_FRAC_PADRAO
CALIB_SPLIT_SEED   = int(manifesto_c5.get("calib_split_seed", C.SEED)) if manifesto_c5 else int(C.SEED)

X_val_cal, y_val_cal = X_val, y_val
validacao_meas       = validacao
if HONEST_CALIB_SPLIT:
    idx_cal, idx_meas = _reproduzir_split_val_cal(y_val, CALIB_SPLIT_FRAC, CALIB_SPLIT_SEED)
    hash_local = hashlib.sha256(idx_cal.tobytes()).hexdigest()[:HASH_LEN]
    hash_c5    = manifesto_c5.get("calib_split_hash") if manifesto_c5 else None
    if hash_c5 and hash_local != hash_c5:
        raise RuntimeError(f"Split divergente do notebook 10 (local={hash_local}, notebook 10={hash_c5}).")
    X_val_cal = X_val.iloc[idx_cal].reset_index(drop=True)
    y_val_cal = y_val.iloc[idx_cal].reset_index(drop=True)
    validacao_meas = validacao.iloc[idx_meas].reset_index(drop=True)

if finalista_escolhido not in finalistas_aceitos():
    raise ValueError(
        f"Finalista desconhecido: {finalista_escolhido!r}. "
        f"Aceitos: {list(finalistas_aceitos())}."
    )

pipe_base = build_finalista(finalista_escolhido)
pipe_base.fit(X_train, y_train)

if calib_ativa:
    if calib_metodo is None:
        raise ValueError("calib_metodo deve ser definido quando a calibração está ativa.")
    calibrador = calibrar(pipe_base, X_val_cal, y_val_cal, method=calib_metodo)
else:
    calibrador = None

preditor = calibrador if calibrador is not None else pipe_base

display(pd.DataFrame({
    "bloco":       ["treino (train_ref)", "calibração (val_cal)", "medição (val_meas, não usada aqui)"],
    "n":           [len(treino), len(X_val_cal), len(validacao_meas)],
    "positivos":   [int(y_train.sum()), int(y_val_cal.sum()), int(pd.to_numeric(validacao_meas[ALVO]).sum())],
    "prevalencia": [float(y_train.mean()), float(y_val_cal.mean()), float(pd.to_numeric(validacao_meas[ALVO]).mean())],
}))
print(f"finalista   : {finalista_escolhido} (fábrica: {resolver_finalista(finalista_escolhido)})")
print(f"calibração  : {calib_metodo if calib_ativa else 'desativada'}")
print(f"split notebook 10: {'honesto' if HONEST_CALIB_SPLIT else 'validação inteira'}")

,bloco,n,positivos,prevalencia
0,treino (train_ref),556462,46961,0.084392
1,calibração (val_cal),59621,5032,0.084400
2,"medição (val_meas, não usada aqui)",59622,5032,0.084398


finalista   : boosting (fábrica: boosting)
calibração  : sigmoid
split notebook 10: honesto


## Abrir o teste — predição única

Única execução do notebook 11 que toca no teste. Lê `test_ref` e `test_pandemic` separadamente, prevê em lotes e monta as três janelas do YAML (`avaliacao.janelas`): Teste A, 2020–2022 isolado e Teste B. As previsões são gravadas antes de qualquer análise. Nada aqui reajusta modelo, calibração ou limiar.

In [6]:
def prever_em_lotes(modelo, X, tamanho=TAMANHO_LOTE_PREDICAO):
    partes = [modelo.predict_proba(X.iloc[i:i + tamanho])[:, 1] for i in range(0, len(X), tamanho)]
    return np.concatenate(partes) if partes else np.empty(0, dtype="float64")


COLUNA_ANO = str(C.cfg("periodo", "coluna_ano") or "ano_internacao")
_audit_manifesto = set(manifest.get("audit_columns") or [])
COLUNAS_AUDITORIA = [
    c for c in ("id_registro", "group_id", COLUNA_ANO, "sg_uf", "regiao_residencia", "faixa_etaria", "sexo_categoria")
    if c in _audit_manifesto or c in FEATURES
]

abertura_teste_utc = pd.Timestamp.now(tz="UTC").isoformat()
timestamp = pd.Timestamp.now(tz="UTC").strftime("%Y%m%dT%H%M%SZ")
prefixo   = f"{PREFIXO_BASE}_{manifest['cohort_kind']}_{timestamp}"

blocos_teste = {}
if ENSAIO:
    # Duas metades da val_meas fazem o papel de test_ref e test_pandemic; o teste real não é lido.
    meio = len(validacao_meas) // 2
    partes_ensaio = {
        "test_ref": validacao_meas.iloc[:meio].reset_index(drop=True),
        "test_pandemic": validacao_meas.iloc[meio:].reset_index(drop=True),
    }
    fonte_blocos = lambda bloco: partes_ensaio[bloco]  # noqa: E731
else:
    colunas_leitura = list(dict.fromkeys(FEATURES + COLUNAS_AUDITORIA + [ALVO]))

    def fonte_blocos(bloco):
        try:
            parte, _ = load_supervised_artifact(
                ROOT_DIR, expected_kind=COHORT_KIND, partitions=(bloco,), columns=colunas_leitura,
            )
        except GatePendenteError as exc:
            raise RuntimeError(
                "G9_holdout ficou pendente entre a checagem e o carregamento. "
                "Verifique configs/supervised.yaml antes de rodar novamente."
            ) from exc
        return parte.reset_index(drop=True)

for bloco in BLOCOS_TESTE:
    parte = fonte_blocos(bloco)
    y_bloco = pd.to_numeric(parte[ALVO], errors="coerce").astype("int8").to_numpy()
    prob_bloco = prever_em_lotes(preditor, parte[FEATURES])
    if not np.isfinite(prob_bloco).all() or not ((prob_bloco >= 0) & (prob_bloco <= 1)).all():
        raise RuntimeError(f"Probabilidades inválidas no bloco {bloco}.")
    colunas_guardar = COLUNAS_AUDITORIA + ([c for c in FEATURES if c not in COLUNAS_AUDITORIA] if len(parte) <= LIMITE_LINHAS_COM_FEATURES else [])
    meta = parte[[c for c in colunas_guardar if c in parte.columns]].copy()
    blocos_teste[bloco] = {"meta": meta, "y": y_bloco, "prob": prob_bloco}
    del parte


def montar_janela(blocos, com_features=False):
    partes = [blocos_teste[b] for b in blocos]
    colunas = COLUNAS_AUDITORIA + ([c for c in FEATURES if c not in COLUNAS_AUDITORIA] if com_features else [])
    meta = pd.concat(
        [p["meta"][[c for c in colunas if c in p["meta"].columns]].assign(bloco=b) for p, b in zip(partes, blocos)],
        ignore_index=True,
    )
    return {
        "blocos": list(blocos),
        "meta": meta,
        "y": np.concatenate([p["y"] for p in partes]),
        "prob": np.concatenate([p["prob"] for p in partes]),
    }


# Só a janela principal (Teste A) guarda as variáveis do modelo, para a análise de falsos negativos.
JANELAS_DADOS = {nome: montar_janela(blocos, com_features=(nome == JANELA_PRINCIPAL)) for nome, blocos in JANELAS.items()}

linhas_janelas = []
for nome, j in JANELAS_DADOS.items():
    m = metricas_completas(j["y"], j["prob"], limiar=limiar_final_val)
    prevalencia = float(j["y"].mean())
    ap_janela = float(m.get("pr_auc", m.get("ap")))
    linhas_janelas.append({
        "janela": nome, "blocos": "+".join(j["blocos"]), "n": int(len(j["y"])),
        "positivos": int(j["y"].sum()), "prevalencia": prevalencia,
        "pr_auc_sobre_prevalencia": ap_janela / prevalencia if prevalencia > 0 else float("nan"),
        **{k: float(v) for k, v in m.items() if k != "n"},
    })
janelas_df = pd.DataFrame(linhas_janelas)

principal   = JANELAS_DADOS[JANELA_PRINCIPAL]
y_test_arr  = principal["y"]
prob_test   = principal["prob"]
y_pred_test = (prob_test >= limiar_final_val).astype(int)
metricas_test = metricas_completas(y_test_arr, prob_test, limiar=limiar_final_val)
matriz_test   = matriz_confusao(y_test_arr, y_pred_test)

display(janelas_df)
display(matriz_test)

# Persiste as previsões já, antes de qualquer análise: se algo falhar depois, o resultado não se perde.
def _preparar_para_arquivo(tabela):
    for coluna in tabela.columns:
        if str(tabela[coluna].dtype) in ("object", "category"):
            tabela[coluna] = tabela[coluna].astype("string")
    return tabela


pred_teste = pd.concat(
    [
        pd.DataFrame({
            "bloco": b, "y_true": blocos_teste[b]["y"], "y_prob": blocos_teste[b]["prob"],
            "y_pred": (blocos_teste[b]["prob"] >= limiar_final_val).astype("int8"),
        }).join(blocos_teste[b]["meta"][[c for c in COLUNAS_AUDITORIA if c in blocos_teste[b]["meta"].columns]])
        for b in BLOCOS_TESTE
    ],
    ignore_index=True,
)
pred_teste["limiar"] = limiar_final_val
pred_teste["modelo"] = finalista_escolhido
pred_teste["cohort_kind"] = manifest["cohort_kind"]
pred_teste = _preparar_para_arquivo(pred_teste)

caminho_predicoes = PREDICTIONS_DIR / f"{prefixo}_predicoes_teste.parquet"
try:
    pred_teste.to_parquet(caminho_predicoes, index=False)
except Exception as exc:  # noqa: BLE001
    caminho_predicoes = PREDICTIONS_DIR / f"{prefixo}_predicoes_teste.csv.gz"
    pred_teste.to_csv(caminho_predicoes, index=False, compression="gzip")
    print(f"AVISO: não consegui gravar parquet ({exc!r}); previsões gravadas em {caminho_predicoes.name}.")
del pred_teste

print(f"Teste aberto em UTC: {abertura_teste_utc}")
for nome, j in JANELAS_DADOS.items():
    print(f"  {nome:<18} n={len(j['y']):>10,}  óbitos={int(j['y'].sum()):>9,}")
print(f"Previsões: {caminho_predicoes}")

,janela,blocos,n,positivos,prevalencia,pr_auc_sobre_prevalencia,auc,ap,pr_auc,brier,log_loss,f1,precision,recall,sensibilidade,especificidade,accuracy,limiar
0,teste_a,test_ref,29811,2491,0.083560,4.351642,0.866077,0.363622,0.363622,0.062407,0.211411,0.379469,0.247612,0.811722,0.811722,0.775110,0.778169,0.1178
1,teste_b,test_ref+test_pandemic,59622,5032,0.084398,4.297125,0.866201,0.362670,0.362670,0.063063,0.212566,0.380894,0.249082,0.809022,0.809022,0.775179,0.778035,0.1178
2,pandemico_isolado,test_pandemic,29811,2541,0.085237,4.256739,0.866343,0.362832,0.362832,0.063720,0.213722,0.382312,0.250550,0.806375,0.806375,0.775248,0.777901,0.1178


,pred_0,pred_1,total
real_0,21176,6144,27320
real_1,469,2022,2491
total,21645,8166,29811


Teste aberto em UTC: 2026-11-05T20:59:21.231077+00:00
  teste_a            n=    29,811  óbitos=    2,491
  teste_b            n=    59,622  óbitos=    5,032
  pandemico_isolado  n=    29,811  óbitos=    2,541
Previsões: C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\outputs\predictions\11_ensaio_validacao_final_final_20261105T205921Z_predicoes_teste.parquet


## IC bootstrap, erros e avaliação estratificada

IC percentil por grupo (`group_id`) para AP, AUC e Brier no Teste A; falsos negativos e sua distribuição por região e faixa etária (Teste A); métricas por ano e calibração por ano no Teste B; métricas por grupo no Teste A e no Teste B. Para 2020–2022 e para o Teste B só há estimativa pontual: com cerca de 3 milhões de fichas, o IC seria demorado e estreito demais para informar.

In [7]:
boot_cfg   = C.cfg("avaliacao", "bootstrap") or {}
BOOT_ATIVO = bool(boot_cfg.get("ativo", True))
BOOT_REPS  = int(boot_cfg.get("repeticoes") or 1000)
BOOT_NIVEL = float(boot_cfg.get("nivel") or 0.95)
BOOT_SEED  = int(boot_cfg.get("seed") or C.SEED)
if ENSAIO:
    BOOT_REPS = min(BOOT_REPS, 50)

# IC só na janela principal (Teste A). Bootstrap por grupo (group_id), como pede o protocolo;
# sem group_id, cai no bootstrap por ficha e isso fica registrado na tabela.
grupos_boot = principal["meta"]["group_id"].to_numpy() if "group_id" in principal["meta"].columns else None
IC_METODO   = "bootstrap_por_grupo" if grupos_boot is not None else "bootstrap_por_ficha"

ic_resultados = {}
if BOOT_ATIVO:
    for nome, fn in [("ap", average_precision), ("auc", auc), ("brier", brier_metric)]:
        if grupos_boot is not None:
            r_ic = bootstrap_grupo(
                y_test_arr, prob_test, grupos_boot,
                metrica=nome, n=BOOT_REPS, alpha=1.0 - BOOT_NIVEL, seed=BOOT_SEED,
            )
            est, inf, sup = r_ic["estimativa"], r_ic["ic_inferior"], r_ic["ic_superior"]
        else:
            est, inf, sup = ic_bootstrap(
                y_test_arr, prob_test, metrica=fn,
                n_boot=BOOT_REPS, alpha=1.0 - BOOT_NIVEL, seed=BOOT_SEED,
            )
        ic_resultados[nome] = {
            "estimativa": est, "ic_inferior": inf, "ic_superior": sup,
            "nivel": BOOT_NIVEL, "repeticoes": BOOT_REPS, "metodo": IC_METODO, "janela": JANELA_PRINCIPAL,
        }

ic_df = pd.DataFrame([{"metrica": k, **v} for k, v in ic_resultados.items()])
display(ic_df)

,metrica,estimativa,ic_inferior,ic_superior,nivel,repeticoes,metodo,janela
0,ap,0.363622,0.344645,0.380493,0.95,50,bootstrap_por_grupo,teste_a
1,auc,0.866077,0.860536,0.870452,0.95,50,bootstrap_por_grupo,teste_a
2,brier,0.062407,0.059959,0.064244,0.95,50,bootstrap_por_grupo,teste_a


In [8]:
erros_cfg   = C.cfg("avaliacao", "analise_erros") or {}
ERROS_ATIVA = bool(erros_cfg.get("ativa", True))

# Falsos negativos: só na janela principal (Teste A), onde as variáveis do modelo ficaram guardadas.
df_err = principal["meta"].copy()
df_err["y_true"] = y_test_arr
df_err["y_pred"] = y_pred_test
df_err["y_prob"] = prob_test

fn = df_err[(df_err.y_true == 1) & (df_err.y_pred == 0)]
tp = df_err[(df_err.y_true == 1) & (df_err.y_pred == 1)]
fp = df_err[(df_err.y_true == 0) & (df_err.y_pred == 1)]
tn = df_err[(df_err.y_true == 0) & (df_err.y_pred == 0)]

display(pd.DataFrame([
    {"classe": "FN (óbito→cura)",  "n": len(fn), "frac": len(fn) / len(df_err)},
    {"classe": "TP (óbito→óbito)", "n": len(tp), "frac": len(tp) / len(df_err)},
    {"classe": "FP (cura→óbito)",  "n": len(fp), "frac": len(fp) / len(df_err)},
    {"classe": "TN (cura→cura)",   "n": len(tn), "frac": len(tn) / len(df_err)},
]))

comparacao_fn_tp = pd.DataFrame()
amostra_fn       = pd.DataFrame()
FEATURES_NO_TESTE = [c for c in FEATURES if c in df_err.columns]

if ERROS_ATIVA and len(fn) > 0 and len(tp) > 0:
    feats_num = [c for c in FEATURES_NO_TESTE if pd.api.types.is_numeric_dtype(df_err[c])]
    comparacao_fn_tp = pd.DataFrame([
        {"feature":    c,
         "fn_media":   float(fn[c].mean()),
         "tp_media":   float(tp[c].mean()),
         "delta_media": float(fn[c].mean() - tp[c].mean())}
        for c in feats_num
    ])
    if not comparacao_fn_tp.empty:
        comparacao_fn_tp["abs_delta"] = comparacao_fn_tp.delta_media.abs()
        comparacao_fn_tp = comparacao_fn_tp.sort_values("abs_delta", ascending=False).reset_index(drop=True)
        print("Features numéricas com maior diferença FN vs TP:")
        display(comparacao_fn_tp)

if len(fn) > 0:
    colunas_fn = FEATURES_NO_TESTE + [c for c in ("group_id", COLUNA_ANO, "regiao_residencia") if c in df_err.columns and c not in FEATURES_NO_TESTE]
    amostra_fn = fn.sort_values("y_prob", ascending=False).head(N_FN_AMOSTRA)[colunas_fn + ["y_prob"]]
    print(f"Top {N_FN_AMOSTRA} falsos negativos por probabilidade predita:")
    display(amostra_fn)

# Distribuição dos falsos negativos por região e faixa etária (insumo do relatório).
fn_por_grupo = {}
for grupo in ("regiao_residencia", "faixa_etaria"):
    if grupo in df_err.columns:
        tab = (
            df_err.assign(fn=((df_err.y_true == 1) & (df_err.y_pred == 0)).astype(int), obito=(df_err.y_true == 1).astype(int))
            .groupby(grupo, dropna=False)[["fn", "obito"]].sum().reset_index()
        )
        tab["pct_obitos_sem_alerta"] = tab["fn"] / tab["obito"].where(tab["obito"] > 0)
        fn_por_grupo[grupo] = tab
        display(tab)

,classe,n,frac
0,FN (óbito→cura),469,0.015732
1,TP (óbito→óbito),2022,0.067827
2,FP (cura→óbito),6144,0.206098
3,TN (cura→cura),21176,0.710342


Features numéricas com maior diferença FN vs TP:


,feature,fn_media,tp_media,delta_media,abs_delta
0,idade_anos,32.058372,71.177544,-39.119171,39.119171
1,dias_sintomas_ate_internacao,3.705757,2.925322,0.780435,0.780435


Top 20 falsos negativos por probabilidade predita:


,idade_anos,dias_sintomas_ate_internacao,sexo_categoria,cs_gestant,cs_raca,cs_zona,ave_suino,febre,tosse,garganta,...,neurologic,pneumopati,imunodepre,renal,obesidade,out_morbi,group_id,ano_internacao,regiao_residencia,y_prob
11116,0.002738,0.0,masculino,6,1,1,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,31726497494000,2024,SUDESTE,0.117771
25764,75.000000,2.0,feminino,5,1,1,9,1,1,2,...,2,2,2,2,2,1,31738583098433,2025,SUL,0.117701
10479,62.000000,14.0,feminino,6,4,1,2,1,1,2,...,NaN,NaN,NaN,NaN,NaN,NaN,31726066659013,2024,NORTE,0.117509
28304,80.000000,2.0,masculino,6,1,NaN,NaN,1,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,1,31715881142906,2024,SUDESTE,0.117392
23348,67.000000,7.0,feminino,5,1,1,2,1,1,2,...,2,2,2,2,2,2,31684869552503,2023,CENTRO-OESTE,0.117373
103,57.000000,0.0,feminino,5,1,1,2,2,2,2,...,NaN,NaN,NaN,NaN,NaN,NaN,31725645264088,2024,SUDESTE,0.117224
15087,60.000000,2.0,masculino,6,4,1,9,2,1,2,...,NaN,NaN,NaN,NaN,NaN,NaN,31736514981083,2024,SUDESTE,0.117163
6676,55.000000,7.0,masculino,6,4,1,2,1,1,2,...,2,2,2,2,9,2,315560423240,2019,SUDESTE,0.116899
417,76.000000,0.0,masculino,6,4,1,2,1,1,2,...,2,2,2,2,2,2,31711456452946,2024,SUDESTE,0.116870
3504,75.000000,4.0,feminino,5,4,1,2,2,1,2,...,2,2,2,2,2,2,31721757278632,2024,SUDESTE,0.116197


,regiao_residencia,fn,obito,pct_obitos_sem_alerta
0,NORTE,48,123,0.390244
1,NORDESTE,77,325,0.236923
2,CENTRO-OESTE,41,227,0.180617
3,SUDESTE,212,1260,0.168254
4,SUL,91,556,0.163669
5,AUSENTE,0,0,NaN


,faixa_etaria,fn,obito,pct_obitos_sem_alerta
0,1-4,59,63,0.936508
1,12-17,17,21,0.809524
2,18-39,48,144,0.333333
3,40-59,73,382,0.191099
4,5-11,36,39,0.923077
5,60-79,90,945,0.095238
6,80+,35,778,0.044987
7,<1,111,119,0.932773


In [9]:
metricas_por_ano_df = pd.DataFrame()
curvas_por_ano      = {}
calib_por_ano_df    = pd.DataFrame()

# Por ano: Teste B (test_ref + test_pandemic), para cada ano aparecer uma só vez, de 2019 a 2025.
JANELA_ANUAL = "teste_b" if "teste_b" in JANELAS_DADOS else JANELA_PRINCIPAL
anual = JANELAS_DADOS[JANELA_ANUAL]
if COLUNA_ANO in anual["meta"].columns:
    ano_arr = pd.to_numeric(anual["meta"][COLUNA_ANO], errors="coerce").to_numpy(dtype="float64", na_value=np.nan)
    anos = sorted(int(a) for a in np.unique(ano_arr[~np.isnan(ano_arr)]))
    linhas_ano, linhas_calib = [], []

    for ano in anos:
        m = ano_arr == ano
        y_a, p_a = anual["y"][m], anual["prob"][m]
        if len(y_a) == 0:
            continue
        duas_classes = len(np.unique(y_a)) >= 2
        linhas_ano.append({
            "ano":        int(ano),
            "janela":     JANELA_ANUAL,
            "n":          int(len(y_a)),
            "positivos":  int(y_a.sum()),
            "prevalencia": float(y_a.mean()),
            "ap":   float(average_precision(y_a, p_a)) if duas_classes else float("nan"),
            "auc":  float(auc(y_a, p_a))               if duas_classes else float("nan"),
            "brier": float(brier_metric(y_a, p_a)),
            "ece":  float(ece(y_a, p_a, n_bins=N_BINS_CALIB)) if len(y_a) >= MIN_N_CALIB_ANO else float("nan"),
        })
        if len(y_a) >= MIN_N_CALIB_ANO and duas_classes:
            curvas_por_ano[int(ano)] = curva_confiabilidade(y_a, p_a, n_bins=N_BINS_CALIB)
            d = brier_decomposicao(y_a, p_a, n_bins=N_BINS_CALIB)
            linhas_calib.append({
                "ano": int(ano),
                "n":   int(len(y_a)),
                "ece": float(ece(y_a, p_a, n_bins=N_BINS_CALIB)),
                "mce": float(mce(y_a, p_a, n_bins=N_BINS_CALIB)),
                **{k: float(v) for k, v in d.items()},
            })

    metricas_por_ano_df = pd.DataFrame(linhas_ano)
    calib_por_ano_df    = pd.DataFrame(linhas_calib)

display(metricas_por_ano_df)

# Por grupo: Teste A (resultado principal) e Teste B. Região entra mesmo que não esteja na lista do YAML.
grupos_cfg  = list(dict.fromkeys(list(C.cfg("avaliacao", "grupos") or []) + ["regiao_residencia"]))
min_n_grupo = int(C.cfg("avaliacao", "min_n_grupo") or 100)

metricas_por_grupo_dict = {}
grupos_pulados = []
for nome_janela in dict.fromkeys([JANELA_PRINCIPAL, "teste_b"]):
    if nome_janela not in JANELAS_DADOS:
        continue
    j = JANELAS_DADOS[nome_janela]
    base_grupos = j["meta"].copy()
    if "faixa_etaria" in grupos_cfg and "faixa_etaria" not in base_grupos.columns and "idade_anos" in base_grupos.columns:
        base_grupos["faixa_etaria"] = _faixa_etaria(base_grupos["idade_anos"])
    for grupo in grupos_cfg:
        if grupo not in base_grupos.columns:
            if grupo not in grupos_pulados:
                grupos_pulados.append(grupo)
            continue
        # metricas_por_grupo não aceita valor ausente na coluna de grupo: o ausente vira um grupo "AUSENTE".
        base_grupos[grupo] = base_grupos[grupo].astype("string").fillna("AUSENTE")
        metricas_por_grupo_dict[(nome_janela, grupo)] = metricas_por_grupo(
            base_grupos, j["y"], j["prob"], grupo,
            limiar=limiar_final_val, min_n=min_n_grupo,
        )
        print(f"\n=== {nome_janela} · {grupo} ===")
        display(metricas_por_grupo_dict[(nome_janela, grupo)])
    del base_grupos

if grupos_pulados:
    print(f"\nGrupos pulados (ausentes nas previsões): {grupos_pulados}")

display(calib_por_ano_df)

,ano,janela,n,positivos,prevalencia,ap,auc,brier,ece
0,2019,teste_b,3296,397,0.120449,0.356523,0.815154,0.089409,0.011968
1,2023,teste_b,17968,1692,0.094167,0.410033,0.871359,0.067584,0.006313
2,2024,teste_b,16879,1413,0.083713,0.372173,0.867520,0.062060,0.006042
3,2025,teste_b,21479,1530,0.071232,0.310720,0.866824,0.056027,0.009398



=== teste_a · sg_uf ===


,grupo,n,prevalencia,taxa_positivos,pr_auc,pr_auc_sobre_prev,auc,roc_auc,brier,o_e,f1,precision,recall,suprimido
0,SP,7963.0,0.088032,0.088032,0.422292,4.797021,0.882133,0.882133,0.062648,0.965459,0.396689,0.258301,0.854494,False
1,MG,3416.0,0.077869,0.077869,0.253434,3.254627,0.815341,0.815341,0.066077,0.750979,0.285909,0.174564,0.789474,False
2,PR,3146.0,0.074698,0.074698,0.319771,4.280844,0.850065,0.850065,0.059917,0.767513,0.303797,0.186589,0.817021,False
3,RJ,2095.0,0.130310,0.130310,0.552834,4.242443,0.893614,0.893614,0.082325,1.438246,0.536408,0.401089,0.809524,False
4,RS,1837.0,0.123571,0.123571,0.380419,3.078547,0.848852,0.848852,0.088525,1.101922,0.430736,0.285509,0.876652,False
5,SC,1597.0,0.058860,0.058860,0.336857,5.722983,0.865121,0.865121,0.046614,0.875657,0.334842,0.212644,0.787234,False
6,BA,1170.0,0.059829,0.059829,0.434011,7.254190,0.915766,0.915766,0.042592,1.286643,0.490909,0.360000,0.771429,False
7,DF,1081.0,0.026827,0.026827,0.181821,6.777527,0.847515,0.847515,0.028302,0.437491,0.210526,0.122222,0.758621,False
8,GO,1071.0,0.093371,0.093371,0.343725,3.681289,0.853677,0.853677,0.069767,1.048099,0.405063,0.271186,0.800000,False
9,MS,938.0,0.081023,0.081023,0.306820,3.786799,0.874985,0.874985,0.062206,0.891909,0.384401,0.243816,0.907895,False



=== teste_a · sexo_categoria ===


,grupo,n,prevalencia,taxa_positivos,pr_auc,pr_auc_sobre_prev,auc,roc_auc,brier,o_e,f1,precision,recall,suprimido
0,masculino,15418.0,0.083604,0.083604,0.374957,4.484943,0.873809,0.873809,0.061487,0.986689,0.397529,0.261963,0.823894,False
1,feminino,14389.0,0.083536,0.083536,0.351495,4.207704,0.856940,0.856940,0.063410,0.959882,0.361310,0.233463,0.798669,False
2,ignorado,4.0,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True



=== teste_a · faixa_etaria ===


... (17 outputs left out)


=== teste_b · regiao_residencia ===


,grupo,n,prevalencia,taxa_positivos,pr_auc,pr_auc_sobre_prev,auc,roc_auc,brier,o_e,f1,precision,recall,suprimido
0,SUDESTE,27629.0,0.092150,0.092150,0.397960,4.318637,0.866378,0.866378,0.067068,0.980676,0.387377,0.253166,0.824430,False
1,SUL,13104.0,0.084173,0.084173,0.324030,3.849577,0.856803,0.856803,0.064985,0.896266,0.351845,0.222623,0.838622,False
2,NORDESTE,8534.0,0.077924,0.077924,0.382126,4.903853,0.888526,0.888526,0.056708,1.282860,0.454837,0.327177,0.745865,False
3,CENTRO-OESTE,6734.0,0.069944,0.069944,0.294843,4.215443,0.866306,0.866306,0.055072,0.869803,0.357439,0.228235,0.823779,False
4,NORTE,3604.0,0.067980,0.067980,0.281298,4.137950,0.837638,0.837638,0.055162,1.095056,0.350270,0.238235,0.661224,False
5,AUSENTE,17.0,0.117647,0.117647,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True



Grupos pulados (ausentes nas previsões): ['cs_raca', 'cs_zona']


,ano,n,ece,mce,brier,reliability,resolution,uncertainty
0,2019,3296.0,0.011968,0.734118,0.089409,0.000390,0.016365,0.105941
1,2023,17968.0,0.006313,0.140866,0.067584,0.000178,0.017203,0.085300
2,2024,16879.0,0.006042,0.175418,0.062060,0.000179,0.013964,0.076706
3,2025,21479.0,0.009398,0.187221,0.056027,0.000343,0.009849,0.066158


## Persistir artefatos e registar a execução

In [10]:
caminho_janelas   = TABLES_DIR / f"{prefixo}_janelas.csv"
caminho_metricas  = TABLES_DIR / f"{prefixo}_metricas.csv"
caminho_matriz    = TABLES_DIR / f"{prefixo}_matriz_confusao.csv"
caminho_por_ano   = TABLES_DIR / f"{prefixo}_metricas_por_ano.csv"
caminho_calib_ano = TABLES_DIR / f"{prefixo}_calibracao_por_ano.csv"
caminho_ic        = TABLES_DIR / f"{prefixo}_ic_bootstrap.csv"
caminho_erros     = TABLES_DIR / f"{prefixo}_analise_erros.csv"
caminho_fn        = TABLES_DIR / f"{prefixo}_fn_amostra.csv"
caminho_manifesto = TABLES_DIR / f"{prefixo}_manifesto.json"

janelas_df.to_csv(caminho_janelas, index=False, encoding="utf-8-sig")
pd.DataFrame([{
    "finalista": finalista_escolhido,
    "limiar":    limiar_final_val,
    "janela":    JANELA_PRINCIPAL,
    **{k: float(v) for k, v in metricas_test.items() if k != "limiar"},
}]).to_csv(caminho_metricas, index=False, encoding="utf-8-sig")
matriz_test.to_csv(caminho_matriz, encoding="utf-8-sig")
metricas_por_ano_df.to_csv(caminho_por_ano, index=False, encoding="utf-8-sig")
calib_por_ano_df.to_csv(caminho_calib_ano, index=False, encoding="utf-8-sig")
if not ic_df.empty:            ic_df.to_csv(caminho_ic, index=False, encoding="utf-8-sig")
if not comparacao_fn_tp.empty: comparacao_fn_tp.to_csv(caminho_erros, index=False, encoding="utf-8-sig")
if not amostra_fn.empty:       amostra_fn.to_csv(caminho_fn, index=False, encoding="utf-8-sig")

for (nome_janela, grupo), tab in metricas_por_grupo_dict.items():
    tab.to_csv(TABLES_DIR / f"{prefixo}_por_grupo_{nome_janela}_{grupo}.csv", index=False, encoding="utf-8-sig")
for grupo, tab in fn_por_grupo.items():
    tab.to_csv(TABLES_DIR / f"{prefixo}_fn_por_{grupo}.csv", index=False, encoding="utf-8-sig")
for ano, curva in curvas_por_ano.items():
    curva.to_csv(TABLES_DIR / f"{prefixo}_curva_{ano}.csv", index=False, encoding="utf-8-sig")

manifesto_c6 = {
    "cohort_kind":         manifest["cohort_kind"],
    "ensaio":              ENSAIO,
    "dataset_path":        manifest["dataset_path"],
    "dataset_sha256":      manifest["dataset_sha256"],
    "feature_columns":     FEATURES,
    "target_column":       ALVO,
    "partition_column":    PARTICAO,
    "n_treino":            int(len(treino)),
    "n_validacao":         int(len(validacao)),
    "janelas":             janelas_df.to_dict(orient="records"),
    "janela_principal":    JANELA_PRINCIPAL,
    "n_teste":             int(len(JANELAS_DADOS["teste_b"]["y"])) if "teste_b" in JANELAS_DADOS else int(len(y_test_arr)),
    "finalista_escolhido": finalista_escolhido,
    "calibracao_ativa":    calib_ativa,
    "calibracao_metodo":   calib_metodo,
    "calibracao_fonte":    calib_fonte,
    "honest_calib_split":  HONEST_CALIB_SPLIT,
    "manifesto_c5_usado":  str(manifesto_c5_path) if manifesto_c5_path else None,
    "limiar_final":        limiar_final_val,
    "limiar_metrica":      limiar_metrica,
    "teste_aberto_em_utc": abertura_teste_utc,
    "test_opened":         not ENSAIO,
    "metricas_agregadas":  {k: float(v) for k, v in metricas_test.items()},
    "ic_bootstrap":        ic_resultados,
    "erros":               {"fn": len(fn), "tp": len(tp), "fp": len(fp), "tn": len(tn)},
    "grupos_avaliados":    [f"{j}:{g}" for (j, g) in metricas_por_grupo_dict],
    "grupos_pulados":      grupos_pulados,
    "artefatos": {
        "janelas":            str(caminho_janelas),
        "metricas":           str(caminho_metricas),
        "matriz_confusao":    str(caminho_matriz),
        "metricas_por_ano":   str(caminho_por_ano),
        "calibracao_por_ano": str(caminho_calib_ano),
        "ic_bootstrap":       str(caminho_ic),
        "analise_erros":      str(caminho_erros) if not comparacao_fn_tp.empty else None,
        "fn_amostra":         str(caminho_fn)    if not amostra_fn.empty else None,
        "predicoes_teste":    str(caminho_predicoes),
    },
}
caminho_manifesto.write_text(
    json.dumps(manifesto_c6, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8",
)

with Run(
    modelo=PREFIXO_BASE,
    parametros={
        "cohort_kind":         manifest["cohort_kind"],
        "ensaio":              ENSAIO,
        "finalista":           finalista_escolhido,
        "calibracao_metodo":   calib_metodo,
        "limiar_final":        limiar_final_val,
        "limiar_metrica":      limiar_metrica,
        "teste_aberto_em_utc": abertura_teste_utc,
        "n_acessos_antes":     n_acessos_antes,
    },
    dataset={"n_treino": len(treino), "n_validacao": len(validacao), "n_teste": int(sum(len(v["y"]) for v in blocos_teste.values()))},
    features={"n_features": len(FEATURES), "usadas": FEATURES},
    notas="C6 — abertura única do teste (ensaio: metades da val_meas). Pipeline reconstruído via build_finalista.",
    dataset_path=Path(manifest["dataset_path"]),
    dataset_sha256=manifest["dataset_sha256"],
) as r:
    r.metrica(**{k: float(v) for k, v in metricas_test.items()})

print(f"Manifesto: {caminho_manifesto}")
print(f"Run registrado em: {r.pasta}")

Execução registrada em: C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\outputs\runs\20261105T205923835407Z_11_ensaio_validacao_final
Manifesto: C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\outputs\tables\11_ensaio_validacao_final_final_20261105T205921Z_manifesto.json
Run registrado em: C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\outputs\runs\20261105T205923835407Z_11_ensaio_validacao_final


## Marcar `G9_holdout` como executado

Edição cirúrgica do YAML: só o bloco `G9_holdout` é tocado (preserva as quebras de linha do arquivo). Em seguida registra `test_opened` no manifesto do artefato. Na próxima execução, a verificação de gates e a guarda de reabertura falham — disciplina de abertura única. No ensaio nada disso é alterado.

In [11]:
def _marcar_g9_executado(root_dir: Path, timestamp_utc: str, n_antes: int) -> Path:
    yaml_path = root_dir / "configs" / "supervised.yaml"
    bruto     = yaml_path.read_bytes().decode("utf-8")
    nl        = "\r\n" if "\r\n" in bruto else "\n"
    linhas    = bruto.replace("\r\n", "\n").splitlines(keepends=True)

    i_ini = next(
        (i for i, ln in enumerate(linhas) if ln.rstrip() == "  G9_holdout:"),
        None,
    )
    if i_ini is None:
        raise RuntimeError("Bloco `G9_holdout:` não encontrado em supervised.yaml.")

    indent = len(linhas[i_ini]) - len(linhas[i_ini].lstrip(" "))
    i_fim  = len(linhas)
    for i in range(i_ini + 1, len(linhas)):
        ln = linhas[i]
        if ln.strip() and (len(ln) - len(ln.lstrip(" "))) < indent:
            i_fim = i
            break

    novos = {
        "status":       "executado",
        "executado_em": f'"{timestamp_utc}"',
        "n_acessos":    str(n_antes + 1),
    }
    padrao = re.compile(r"^(\s+)(\w+):\s*.*$")
    encontrados = set()
    bloco = linhas[i_ini:i_fim]
    for j, ln in enumerate(bloco):
        if (m := padrao.match(ln)) and m.group(2) in novos:
            bloco[j] = f"{m.group(1)}{m.group(2)}: {novos[m.group(2)]}\n"
            encontrados.add(m.group(2))

    if faltando := set(novos) - encontrados:
        raise RuntimeError(
            f"Campos ausentes em G9_holdout: {sorted(faltando)}. "
            "O teste JÁ FOI ABERTO — corrija o YAML e defina SRAG_ALLOW_TEST_REOPEN=1 para reexecutar."
        )

    linhas[i_ini:i_fim] = bloco
    yaml_path.write_bytes("".join(linhas).replace("\n", nl).encode("utf-8"))
    return yaml_path


def _registrar_teste_aberto_no_artefato(manifest_dataset_path: str, timestamp_utc: str) -> Path:
    caminho = Path(manifest_dataset_path).parent / MANIFEST_NAME
    dados = json.loads(caminho.read_text(encoding="utf-8"))
    dados["test_opened"] = True
    dados["test_opened_utc"] = timestamp_utc
    dados["test_opened_by"] = "11_validacao_final"
    caminho.write_text(json.dumps(dados, ensure_ascii=False, indent=2), encoding="utf-8")
    return caminho


if ENSAIO:
    caminho_yaml = None
    print("ENSAIO: G9 e o manifesto do artefato NÃO foram alterados.")
else:
    caminho_yaml = _marcar_g9_executado(ROOT_DIR, abertura_teste_utc, n_acessos_antes)
    caminho_manifesto_artefato = _registrar_teste_aberto_no_artefato(manifest["dataset_path"], abertura_teste_utc)
    print(f"G9 marcado em {caminho_yaml}")
    print(f"test_opened registrado em {caminho_manifesto_artefato}")

print()
print("=" * 60)
print(f"notebook 11{' (ENSAIO)' if ENSAIO else ''} concluído. AP={metricas_test['ap']:.4f}  AUC={metricas_test['auc']:.4f}  Brier={metricas_test['brier']:.4f}")
print(f"Falsos negativos: {len(fn)} de {int(y_test_arr.sum())} óbitos ({JANELA_PRINCIPAL})")
print("=" * 60)

ENSAIO: G9 e o manifesto do artefato NÃO foram alterados.

notebook 11 (ENSAIO) concluído. AP=0.3636  AUC=0.8661  Brier=0.0624
Falsos negativos: 469 de 2491 óbitos (teste_a)
